# Decision economics

A model score is not a decision. A lender has to say yes or no to each loan, and the score is
only worth something once it is turned into that choice and the choice into money. The rule that
does the turning has to know what each outcome is worth: what a repaid loan earns, and what a
default costs.

Those two are not symmetric and do not scale together, so a loan's break-even probability is a
property of that loan rather than one number for the book. This notebook prices the decision that
way and asks what it is worth in currency rather than AUC.

What it does:

- Compares three decision rules, coarsest to finest: approve everything, one break-even threshold
  for the whole book, and a per-loan bar that rises and falls with each loan's rate.
- Scores each on what the loans actually repaid, in money, not on the margin the model assumed,
  so no rule is graded on its own predictions.
- Prices a loan from two constants, the margin earned per point of rate and the share of
  principal lost on default, calibrated on the training vintages and checked against the database
  before use.

In [10]:
from pathlib import Path

import duckdb
import pandas as pd

from credit_risk.data import (
    load_loans,
    load_outcomes,
    DB_PATH
)
from credit_risk.split import out_of_time_split
from credit_risk.model import (
    build_lgbm,
    LC_VERDICT_NUMERIC, LC_VERDICT_CATEGORICAL,
    UNDERWRITER_NUMERIC, UNDERWRITER_CATEGORICAL,
)
from credit_risk.evaluate import (
    expected_profit,
    breakeven_probability,
    MARGIN_PER_RATE_POINT,
    LOSS_FRACTION,
)

TARGET = "target_bad"
NUMERIC = UNDERWRITER_NUMERIC + LC_VERDICT_NUMERIC
CATEGORICAL = UNDERWRITER_CATEGORICAL + LC_VERDICT_CATEGORICAL

df = load_loans()
train, val, _ = out_of_time_split(df)

cols = NUMERIC + CATEGORICAL
pipe = build_lgbm(NUMERIC, CATEGORICAL)
pipe.fit(train[cols], train[TARGET])

proba=pipe.predict_proba(val[cols])
val = val.assign(proba=proba[:, 1])

print(f"validation {len(val)} loans, bad rate {val[TARGET].mean():.3f}")

validation 154703 loans, bad rate 0.150


## What each outcome is worth

`evaluate.py` carries the margin slope and the loss fraction as defaults, so the rule works
without a database. They were calibrated in `sql/30_loan_economics.sql` on the training vintages
only, from payment columns that exist only after origination: fine for setting business
parameters, leakage if those columns were fed to the model.

Written down like that they can quietly go stale, so the cell below reads them back from the
database and prints both side by side. If they ever diverge, it shows here rather than sitting
unnoticed in a constant.

In [11]:
# Re-run the calibration query and compare it against the constants the rule ships with.
# The file holds two queries; the constants are the last one.
statements = Path("../sql/30_loan_economics.sql").read_text().split(";")
economics_sql = [s for s in statements if s.strip()][-1]

with duckdb.connect(str(DB_PATH), read_only=True) as con:
    calibrated = con.execute(economics_sql).df().iloc[0]

pd.DataFrame({
    "from sql": {
        "margin_per_rate_point": calibrated["margin_per_rate_point"],
        "loss_fraction": calibrated["loss_fraction"],
    },
    "in evaluate.py": {
        "margin_per_rate_point": MARGIN_PER_RATE_POINT,
        "loss_fraction": LOSS_FRACTION,
    },
})

,from sql,in evaluate.py
margin_per_rate_point,0.0133,0.0133
loss_fraction,0.3543,0.3543


## Three policies

Three rules for turning the score into a yes or no, scored on the same validation loans:

- **Approve all.** Fund every loan. The floor any rule has to beat, and the yardstick for how
  much a decision is even worth on this book.
- **Single break-even.** One threshold for the whole book: approve when the predicted default is
  below the break-even probability of the average loan. Derived from the economics rather than
  tuned on profit, which would be the mirror image of an arbitrary cutoff: the amount-weighted
  average rate, taken on the training vintages, so the baseline never sees what it is judged on.
- **Expected profit.** Approve when the loan's own expected profit is positive, so the bar rises
  and falls with its rate rather than sitting at one number for everyone.

Each loan is paid what it actually paid, not what the margin slope predicts. Otherwise the
profit rule would be settling its own bill.

In [12]:
# Realised outcome per loan. val already has loan_amnt, so drop it from the outcomes side and
# keep the payment columns only.
outcomes = load_outcomes().drop(columns=["loan_amnt"])
scored = val.merge(outcomes, on="id")
assert len(scored) == len(val)   # every validation loan should find its outcome

# Principal back, plus recoveries and interest, minus what was lent. Repaid loans return all
# their principal so this collapses to the interest earned; charged-off ones come out negative
# on their own. No branch on target_bad, so no sign to get wrong.
scored["realised_profit"] = (
    scored["total_rec_prncp"] + scored["recoveries"] + scored["total_rec_int"] - scored["loan_amnt"]
)

scored["exp_profit"] = expected_profit(scored["proba"], scored["loan_amnt"], scored["int_rate"])

# The single threshold is the per-loan rule with the rate collapsed to one number: the break-even
# of the book's average loan. Weight the rate by amount, since profit follows the amounts, and
# take it on train so the baseline never touches validation.
r_bar = (train["int_rate"] * train["loan_amnt"]).sum() / train["loan_amnt"].sum()
c = breakeven_probability(r_bar)
print(f"single break-even: {c:.3f}, from an average rate of {r_bar:.1f}%")

policies = {
    "approve all": pd.Series(True, index=scored.index),
    "single break-even": scored["proba"] < c,
    "expected profit": scored["exp_profit"] > 0,
}

rows = {}
for name, approve in policies.items():
    taken = scored[approve]
    rows[name] = {
        "approved": len(taken),
        "total_profit": taken["realised_profit"].sum(),
        "profit_per_loan": taken["realised_profit"].mean(),
        "bad_rate": taken[TARGET].mean(),
    }

pd.DataFrame(rows).T

single break-even: 0.316, from an average rate of 12.3%


,approved,total_profit,profit_per_loan,bad_rate
approve all,154703.0,1.321967e+08,854.519481,0.150016
single break-even,151114.0,1.331666e+08,881.232924,0.144308
expected profit,154319.0,1.324130e+08,858.047016,0.149405


In [13]:
# The two model-based rules mostly agree. Where they part is the whole story, so look at the loans
# expected profit keeps and the single threshold turns away.
be = scored["proba"] < c
ep = scored["exp_profit"] > 0

print(f"single break-even rejects {(~be).sum()}, expected profit rejects {(~ep).sum()}")

extra = scored[ep & ~be]   # kept only by the per-loan rule
print(f"kept only by expected profit: {len(extra)} loans at {extra['int_rate'].mean():.0f}% average rate, "
      f"{extra[TARGET].mean():.0%} default, {extra['realised_profit'].sum() / 1e6:.1f}M realised")

single break-even rejects 3589, expected profit rejects 384
kept only by expected profit: 3258 loans at 19% average rate, 39% default, -0.8M realised


## Assumptions

No discounting, so a euro at month 36 counts as a euro today. Prepayment is not modelled; loans
repaid early earn less interest, and that is already inside the realised margin rather than
handled separately. And the economics are taken from past loans, so the figures hold only as
long as pricing and recovery behave as they did.

## Conclusions

The three policies finish within 1% of each other, between 132.2 and 133.2M on the validation
book. On a portfolio already screened down to funded loans, at rates where the interest covers
the risk almost everywhere, the approve-or-reject decision barely moves the total. Approving
everything is already close to the best on offer.

The single break-even edges ahead, 133.2M against 132.4M for per-loan pricing, and that is the
result worth pausing on: pricing each loan on its own rate does not beat one threshold for the
whole book. The reason sits in the loans the two rules disagree on. Per-loan break-even rises
with the rate, reaching 0.48 at the top, so the rule keeps 3,258 high-rate loans the flat cutoff
turns away, at 19% average rate. Those loans default near 40% and lose money in aggregate: the
margin the model priced them on overstated what they actually returned. A single, lower threshold
never reaches for them.

So the per-loan rule's appeal, a bar that scales with the rate, is also where it slips. It trusts
the calibrated margin far out on the rate scale, which is exactly where the fit is thinnest and
the defaults heaviest.

Where this stops holding: the book is only Lending Club's approved loans, already filtered, so
approving everything looks stronger than it would against real applicants, the rejects never
appear. And the figures assume no discounting and past recovery behaviour, per the assumptions
above. The steady reading across all of it is that the decision layer adds little on this book;
the model earns its keep in ranking and pricing, not in the yes-or-no cut.